#**Train / Validation / Test Split**

### **Objective**
Split `loan_data_sampled.csv` into **three** sets instead of two:

- **Train (70%)** — used to fit the model
- **Validation (15%)** — used to compare models and tune hyperparameters
- **Test (15%)** — touched only **once**, at the very end, for the final
  evaluation  



**1-Import Libraries**

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin


**2-Load Selected Dataset**

In [18]:
df = pd.read_csv("loan_data_sampled.csv")
print(df.shape)
df.head()


(50000, 32)


,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,rate_of_interest,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,cf,NaN,nopre,type2,p4,l1,nopc,b/c,236500,3.250,...,CRIF,758,CIB,45-54,to_inst,82.118056,south,direct,0,NaN
1,cf,Male,nopre,type1,p3,l2,nopc,nob/c,176500,4.375,...,CRIF,761,CIB,45-54,to_inst,65.858209,south,direct,0,36.0
2,cf,Joint,nopre,type1,p1,l1,nopc,nob/c,446500,3.875,...,CRIF,778,EXP,35-44,not_inst,80.017921,south,direct,0,36.0
3,cf,Joint,pre,type2,p1,l1,nopc,b/c,296500,3.875,...,CRIF,868,EXP,35-44,to_inst,99.496644,north,direct,0,43.0
4,cf,Joint,nopre,type1,p4,l1,nopc,nob/c,456500,3.750,...,EXP,678,EXP,55-64,not_inst,41.200361,north,direct,0,22.0


**3-Split Features (X) and Target (y)**

In [19]:
target = "Status"

X = df.drop(columns=[target])
y = df[target]

print(X.shape, y.shape)


(50000, 31) (50000,)


**4-Two-Stage Stratified Split (70 / 15 / 15)**



In [20]:
# 70% train vs 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y,test_size=0.30,stratify=y,random_state=42)

# split the 30% temp into 15% val + 15% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp,test_size=0.50,stratify=y_temp,random_state=42)

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)


Train: (35000, 31)
Val:   (7500, 31)
Test:  (7500, 31)


In [21]:
print("Full dataset:")
print(y.value_counts(normalize=True) * 100)

print("\nTrain:")
print(y_train.value_counts(normalize=True) * 100)

print("\nValidation:")
print(y_val.value_counts(normalize=True) * 100)

print("\nTest:")
print(y_test.value_counts(normalize=True) * 100)


Full dataset:
Status
0    75.356
1    24.644
Name: proportion, dtype: float64

Train:
Status
0    75.357143
1    24.642857
Name: proportion, dtype: float64

Validation:
Status
0    75.346667
1    24.653333
Name: proportion, dtype: float64

Test:
Status
0    75.36
1    24.64
Name: proportion, dtype: float64


In [22]:
train_idx = set(X_train.index)
val_idx = set(X_val.index)
test_idx = set(X_test.index)

print("Train-Val overlap:", len(train_idx & val_idx))
print("Train-Test overlap:", len(train_idx & test_idx))
print("Val-Test overlap:", len(val_idx & test_idx))



Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


**7-Save the Three Splits**

In [23]:
X_train.to_csv("X_train.csv", index=False)
X_val.to_csv("X_val.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

# **Preprocessing Pipeline**

### **Objective**
Apply Missing Value Imputation, Encoding, and Scaling —
everything is **fit on `X_train` only**, then applied (`transform`) to
`X_train`, `X_val`, and `X_test`, so nothing from val/test leaks into the
statistics the model is built on.



**1-Load Train / Val / Test Splits**

In [37]:
X_train = pd.read_csv("X_train.csv")
X_val   = pd.read_csv("X_val.csv")
X_test  = pd.read_csv("X_test.csv")
#squeeze() : converts dataframe to series
y_train = pd.read_csv("y_train.csv").squeeze()
y_val   = pd.read_csv("y_val.csv").squeeze()
y_test  = pd.read_csv("y_test.csv").squeeze()

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Train: (35000, 31) Val: (7500, 31) Test: (7500, 31)


**2-Define Column Groups**



In [38]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
ordinal_features = ["age"] if "age" in X_train.columns else []
age_order = ["<25", "25-34", "35-44", "45-54", "55-64", "65-74", ">74"]
nominal_features = [
    c for c in X_train.select_dtypes(include="object").columns
    if c not in ordinal_features
]

print("Numerical:", numerical_features)
print("Ordinal:", ordinal_features)
print("Nominal:", nominal_features)


Numerical: ['loan_amount', 'rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'term', 'property_value', 'income', 'Credit_Score', 'LTV', 'dtir1']
Ordinal: ['age']
Nominal: ['loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose', 'Credit_Worthiness', 'open_credit', 'business_or_commercial', 'Neg_ammortization', 'interest_only', 'lump_sum_payment', 'construction_type', 'occupancy_type', 'Secured_by', 'total_units', 'credit_type', 'co-applicant_credit_type', 'submission_of_application', 'Region', 'Security_Type']


**3-Handling Missing Values , Encoding and scaling**

In [39]:
# Numerical columns
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])
# Age (Ordinal)
ordinal_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OrdinalEncoder(categories=[age_order]))])
# Categorical columns (Nominal)
nominal_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder( handle_unknown="ignore",sparse_output=False))])
# Combine everything
preprocessor = ColumnTransformer([("num", numeric_pipeline, numerical_features),("ord", ordinal_pipeline, ordinal_features),("nom", nominal_pipeline, nominal_features)])


**4-Fit on Train Only, Transform All Three Sets**

In [40]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Val:  ", X_val_processed.shape)
print("Test: ", X_test_processed.shape)


Train: (35000, 62)
Val:   (7500, 62)
Test:  (7500, 62)


**5-Rebuild Readable DataFrames**

In [41]:
feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_val_processed   = pd.DataFrame(X_val_processed, columns=feature_names, index=X_val.index)
X_test_processed  = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

X_train_processed.head()


,num__loan_amount,num__rate_of_interest,num__Interest_rate_spread,num__Upfront_charges,num__term,num__property_value,num__income,num__Credit_Score,num__LTV,num__dtir1,...,nom__co-applicant_credit_type_CIB,nom__co-applicant_credit_type_EXP,nom__submission_of_application_not_inst,nom__submission_of_application_to_inst,nom__Region_central,nom__Region_north,nom__Region_north-east,nom__Region_south,nom__Security_Type_Indirect,nom__Security_Type_direct
0,-0.189927,-0.090555,-0.204873,0.539864,0.427849,0.016672,-0.520808,-0.581389,-0.335107,0.218025,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1.476312,-0.090555,-0.007318,-0.779199,0.427849,0.937317,0.138636,0.586944,0.032036,0.425940,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,-0.781173,-1.094061,-1.052307,-0.544574,0.427849,-0.702582,-0.660128,0.861846,0.058614,-0.613633,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
3,0.777567,-0.090555,-1.041357,1.440484,0.427849,0.707156,-0.056411,0.458084,-0.208716,0.633854,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.025072,-0.090555,-0.085535,-0.167095,0.427849,-0.213489,-0.474369,-0.375213,0.055926,0.114068,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


**6- Check No Missing Values Left**

In [42]:
print("Train missing:", X_train_processed.isnull().sum().sum())
print("Val missing:  ", X_val_processed.isnull().sum().sum())
print("Test missing: ", X_test_processed.isnull().sum().sum())


Train missing: 0
Val missing:   0
Test missing:  0


**9-Save Processed Data and the Fitted Pipeline**

In [43]:
X_train_processed.to_csv("X_train_processed.csv", index=False)
X_val_processed.to_csv("X_val_processed.csv", index=False)
X_test_processed.to_csv("X_test_processed.csv", index=False)

joblib.dump(preprocessor,"preprocessor.pkl")

print("Saved processed data and preprocessor.pkl")


Saved processed data and preprocessor.pkl
